In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
path = "/content/drive/MyDrive/movie1M_data.csv"
df = pd.read_csv(path)

In [ ]:
df.head()

,title,overview,url,genre
0,Freddy,The lines between love and obsession blur in t...,https://res.cloudinary.com/fj9gmyig/image/uplo...,"Drama, Mystery, Thriller"
1,An Action Hero,Youth Icon. Superstar. Action Hero. At the age...,https://res.cloudinary.com/fj9gmyig/image/uplo...,Action
2,Kantara,It involves culture of Kambala and Bhootha Kol...,https://res.cloudinary.com/fj9gmyig/image/uplo...,"Action, Adventure, Drama"
3,Khakee: The Bihar Chapter,As a righteous cop pursues a merciless crimina...,https://res.cloudinary.com/fj9gmyig/image/uplo...,"Action, Crime, Drama"
4,Drishyam 2,A gripping tale of an investigation and a fami...,https://res.cloudinary.com/fj9gmyig/image/uplo...,"Crime, Drama, Mystery"


In [ ]:
df.shape

(1000590, 4)

In [ ]:
df = df[["title", "overview"]]

print(df.head())
print(df.columns)

                       title  \
0                     Freddy   
1             An Action Hero   
2                    Kantara   
3  Khakee: The Bihar Chapter   
4                 Drishyam 2   

                                            overview  
0  The lines between love and obsession blur in t...  
1  Youth Icon. Superstar. Action Hero. At the age...  
2  It involves culture of Kambala and Bhootha Kol...  
3  As a righteous cop pursues a merciless crimina...  
4  A gripping tale of an investigation and a fami...  
Index(['title', 'overview'], dtype='object')


In [ ]:
df.head()

,title,overview
0,Freddy,The lines between love and obsession blur in t...
1,An Action Hero,Youth Icon. Superstar. Action Hero. At the age...
2,Kantara,It involves culture of Kambala and Bhootha Kol...
3,Khakee: The Bihar Chapter,As a righteous cop pursues a merciless crimina...
4,Drishyam 2,A gripping tale of an investigation and a fami...


In [ ]:
df.to_csv("/content/drive/MyDrive/rag_dataset.csv", index=False)

In [ ]:
df.isnull().sum()

,0
title,0
overview,0
url,0
genre,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df["url"] = df["url"].apply(
    lambda x: "https://image.tmdb.org/t/p/w500" + str(x).strip().strip("'\"")
    if str(x).strip().startswith("/")
    else str(x).strip().strip("'\"")
)

In [ ]:
placeholder_url = "https://res.cloudinary.com/fj9gmyig/image/upload/v1784981983/noposter_clju4t.jpg"

df["url"] = df["url"].replace(
    "https://via.placeholder.com/300x450?text=No+Poster",
    placeholder_url
)

In [ ]:
print(df.loc[43015, "url"])

https://image.tmdb.org/t/p/w500/4Pld6ahonT4jTem3FhY7hniNjPI.jpg


In [ ]:
count = df["url"].astype(str).str.startswith(
    "https://res.cloudinary.com/fj9gmyig/image/upload"
).sum()

print(count)

196144


In [ ]:
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 8.3 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

In [ ]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
documents = ("Title:" + df['title'].astype(str) +
             "\nGenre:" + df['genre'].astype(str) +
             "\nOverview:" + df["overview"].astype(str)).tolist()

In [ ]:
import numpy as np

save_path = "/content/drive/MyDrive/movie_embeddings.npy"

embeddings = model.encode(
    documents,
    batch_size=512,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

np.save(save_path, embeddings)

print("Embeddings saved successfully!")
print("Shape:", embeddings.shape)

Batches:   0%|          | 0/1955 [00:00<?, ?it/s]

Embeddings saved successfully!
Shape: (1000590, 384)


In [ ]:
import faiss

index = faiss.IndexFlatIP(embeddings.shape[1])

index.add(embeddings)

faiss.write_index(
    index,
    "/content/drive/MyDrive/movie_index.faiss"
)

print("FAISS index saved successfully!")
print("Total vectors:", index.ntotal)

FAISS index saved successfully!
Total vectors: 1000590


In [ ]:
faiss.write_index(index, "/content/drive/MyDrive/movie_index.faiss")

In [ ]:
from IPython.display import display, Image

def recommend_movies(movie_name):

    movie = df[df["title"].str.lower() == movie_name.lower()].iloc[0]

    text = (
        "Title: " + str(movie["title"]) +
        "\nGenre: " + str(movie["genre"]) +
        "\nOverview: " + str(movie["overview"])
    )

    query = model.encode(
        [text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    distances, indices = index.search(query, 11)

    results = df.iloc[indices[0][1:31]]

    movie_list = results["title"].tolist()

    print("Recommended Movies:")
    print(movie_list)

    for _, movie in results.iterrows():
        print(movie["title"])
        display(Image(url=movie["url"]))

    return movie_list

In [ ]:
movie_list = recommend_movies("Source Code")

Recommended Movies:
['The Engineer', 'Mobius', 'The Secret Agent', 'Time Pulse', 'Time Bomb', 'Space Terror', 'The Night of the Full Moon', 'The Ticking Man', 'The Clock Is Ticking', 'Man on the Tracks']
The Engineer


Mobius


The Secret Agent


Time Pulse


Time Bomb


Space Terror


The Night of the Full Moon


The Ticking Man


The Clock Is Ticking


Man on the Tracks


In [ ]:
movie_list

NameError: name 'movie_list' is not defined

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 40.2 MB/s eta 0:00:00


In [ ]:
import faiss

path = "/content/drive/MyDrive/movie_index.faiss"

index = faiss.read_index(path)

print("Total vectors:", index.ntotal)


Total vectors: 1839269


In [ ]:
print("Vector dimension:", index.d)

Vector dimension: 384
